# Notebook 1bis — Visualisation du dataset lisible (T / H / P)

## Position dans le pipeline

`NB1 → **NB1bis (optionnel)** → NB2 → NB3 → NB4`  
Guide : `00_PIPELINE_PAS_A_PAS.md`



Fichiers : `data/readable/opensensemap_apercu_2000.csv` ou `opensensemap_lisible_large.parquet`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
READABLE_DIR = PROJECT_ROOT / "data" / "readable"
FIG_DIR = PROJECT_ROOT / "reports" / "figures_exploration"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Aperçu rapide (léger) ou version complète
USE_FULL = False  # True pour opensensemap_lisible_large.parquet

if USE_FULL:
    path = READABLE_DIR / "opensensemap_lisible_large.parquet"
    df = pd.read_parquet(path)
else:
    path = READABLE_DIR / "opensensemap_apercu_2000.csv"
    df = pd.read_csv(path)

df["date_heure_utc"] = pd.to_datetime(df["date_heure_utc"], utc=True, errors="coerce")
print("Fichier :", path)
print("Lignes  :", len(df))
print("Stations:", df["nom_station"].nunique())
df.head()


## 1. Aperçu tabulaire

In [ ]:
display(df[["nom_station", "date_heure_utc", "temperature_C", "humidite_pct", "pression_hPa"]].head(15))
display(df[["temperature_C", "humidite_pct", "pression_hPa"]].describe())


## 2. Séries temporelles pour une station

In [ ]:
# Choisir une station (modifiez le nom si besoin)
station = df["nom_station"].value_counts().index[0]
sub = df[df["nom_station"] == station].sort_values("date_heure_utc")

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)

axes[0].plot(sub["date_heure_utc"], sub["temperature_C"], color="#B85C38", linewidth=1)
axes[0].set_ylabel("Température (°C)")
axes[0].set_title(f"Station : {station}")
axes[0].grid(True, alpha=0.3)

axes[1].plot(sub["date_heure_utc"], sub["humidite_pct"], color="#4C6A80", linewidth=1)
axes[1].set_ylabel("Humidité (%)")
axes[1].grid(True, alpha=0.3)

axes[2].plot(sub["date_heure_utc"], sub["pression_hPa"], color="#2F4F3E", linewidth=1)
axes[2].set_ylabel("Pression (hPa)")
axes[2].set_xlabel("Date (UTC)")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
out = FIG_DIR / "viz_series_temporelles_une_station.png"
fig.savefig(out, dpi=180, bbox_inches="tight")
plt.show()
print("Figure sauvegardée :", out)


## 3. Distributions T / H / P

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))

axes[0].hist(df["temperature_C"].dropna(), bins=30, color="#B85C38", edgecolor="black")
axes[0].set_title("Température (°C)")

axes[1].hist(df["humidite_pct"].dropna(), bins=30, color="#4C6A80", edgecolor="black")
axes[1].set_title("Humidité (%)")

axes[2].hist(df["pression_hPa"].dropna(), bins=30, color="#2F4F3E", edgecolor="black")
axes[2].set_title("Pression (hPa)")

for ax in axes:
    ax.grid(True, alpha=0.25)

plt.tight_layout()
out = FIG_DIR / "viz_distributions_THP.png"
fig.savefig(out, dpi=180, bbox_inches="tight")
plt.show()
print("Figure sauvegardée :", out)


## 4. Carte des stations

In [ ]:
geo = (
    df.groupby(["id_station", "nom_station"], as_index=False)
    .agg(
        latitude=("latitude", "median"),
        longitude=("longitude", "median"),
        temperature_moy=("temperature_C", "mean"),
    )
)

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(
    geo["longitude"],
    geo["latitude"],
    c=geo["temperature_moy"],
    cmap="coolwarm",
    s=60,
    edgecolor="black",
    linewidth=0.4,
)
fig.colorbar(sc, ax=ax, label="Température moyenne (°C)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Stations (aperçu) — température moyenne")
ax.grid(True, alpha=0.3)

plt.tight_layout()
out = FIG_DIR / "viz_carte_stations.png"
fig.savefig(out, dpi=180, bbox_inches="tight")
plt.show()
print("Figure sauvegardée :", out)
display(geo.head(10))
